# Spatial Experiment — Target Detection via Score Matching

**Section 5**: CF-Attn + NeighborMLP vs THANTD/GMM on Pavia-U.

Workflow:
1. Clone repo from GitHub
2. Mount Drive (data file + results only)
3. Dry-run → full run
4. Display figures inline

In [ ]:
# ── Cell 1: Clone repo + install deps ────────────────────────────────────
import subprocess, sys, os

REPO_URL     = 'https://github.com/michaelpiro/final-paper-experiment.git'
LOCAL_PROJECT = '/content/proj'

if os.path.exists(LOCAL_PROJECT):
    # Pull latest changes if already cloned
    subprocess.run(['git', '-C', LOCAL_PROJECT, 'pull'], check=True)
    print('Pulled latest changes.')
else:
    subprocess.run(['git', 'clone', REPO_URL, LOCAL_PROJECT], check=True)
    print(f'Cloned to {LOCAL_PROJECT}')

# Install any missing packages
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'scikit-learn', 'scipy', 'tqdm', 'matplotlib', 'pyyaml'],
               check=True)

# Add to Python path
for path in [LOCAL_PROJECT,
             os.path.join(LOCAL_PROJECT, 'experiments', 'spatial')]:
    if path not in sys.path:
        sys.path.insert(0, path)

os.chdir(LOCAL_PROJECT)
print(f'CWD: {os.getcwd()}')
print('Dependencies OK')

In [ ]:
# ── Cell 2: Mount Drive (data file + results) ─────────────────────────────
# The repo has all code. Drive is only needed for:
#   - pavia-u.mat  (large data file, not in repo)
#   - saving results back to Drive
from google.colab import drive
import os, shutil

drive.mount('/content/drive')

# ---- Symlink / copy the data file ----
DRIVE_DATA  = '/content/drive/MyDrive/final_paper/pavia-u.mat'
LOCAL_DATA  = '/content/proj/data/pavia-u.mat'

os.makedirs('/content/proj/data', exist_ok=True)

if not os.path.exists(LOCAL_DATA):
    if os.path.exists(DRIVE_DATA):
        os.symlink(DRIVE_DATA, LOCAL_DATA)
        print(f'Symlinked data: {DRIVE_DATA} → {LOCAL_DATA}')
    else:
        # Also check the old real_datasets path inside the zip
        alt = '/content/drive/MyDrive/final_paper/real_datasets/pavia-u.mat'
        if os.path.exists(alt):
            os.symlink(alt, LOCAL_DATA)
            print(f'Symlinked data from alt path.')
        else:
            raise FileNotFoundError(
                f'pavia-u.mat not found on Drive.\n'
                f'Upload it to /MyDrive/final_paper/pavia-u.mat')
else:
    print('Data file already present.')

RESULTS_DIR = '/content/drive/MyDrive/final_paper/spatial_results'
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f'Results will be saved to: {RESULTS_DIR}')

In [ ]:
# ── Cell 3a: DRY RUN — verify pipeline before full run ───────────────────
import subprocess, sys

RESULTS_DIR = '/content/drive/MyDrive/final_paper/spatial_results'

result = subprocess.run(
    [sys.executable, '-u',
     'experiments/spatial/run_colab.py',
     '--config', 'experiments/spatial/colab.yaml',
     '--results_dir', RESULTS_DIR,
     '--dry-run',
     '--no-thantd'],
    capture_output=True, text=True
)
# Print full output so nothing is hidden
print(result.stdout)
if result.stderr:
    print('STDERR:', result.stderr[-3000:])
print('Exit code:', result.returncode)
if result.returncode != 0:
    print('DRY RUN FAILED — fix errors before full run.')
else:
    print('DRY RUN PASSED ✓ — proceed to Cell 3b.')

In [ ]:
# ── Cell 3b: FULL RUN (~2-3 h on T4) ─────────────────────────────────────
# Checkpoints saved after each model — safe to restart on disconnect.
import subprocess, sys

RESULTS_DIR = '/content/drive/MyDrive/final_paper/spatial_results'

result = subprocess.run(
    [sys.executable, '-u',
     'experiments/spatial/run_colab.py',
     '--config', 'experiments/spatial/colab.yaml',
     '--results_dir', RESULTS_DIR,
     # '--no-thantd',  # uncomment to skip THANTD
    ],
    capture_output=False, text=True
)
print('Exit code:', result.returncode)

In [ ]:
# ── Cell 4: Display figures inline ───────────────────────────────────────
import os, glob, json
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import subprocess as sp

RESULTS_DIR = '/content/drive/MyDrive/final_paper/spatial_results'

runs = sorted(glob.glob(os.path.join(RESULTS_DIR, 'colab_*')))
if not runs:
    print('No results found. Run Cell 3 first.')
else:
    run_dir = runs[-1]
    print(f'Showing: {run_dir}')

    # AUC summary
    mpath = os.path.join(run_dir, 'all_metrics.json')
    if os.path.exists(mpath):
        all_metrics = json.load(open(mpath))
        print('\n=== AUC Summary (additive) ===')
        for sid_key, sid_data in all_metrics.items():
            for bkey, bdata in sid_data.items():
                if not bkey.startswith('n'): continue
                auc = bdata.get('additive', {}).get('auc', {})
                vals = '  '.join(f'{k}={v:.3f}' for k,v in auc.items() if isinstance(v, float))
                print(f'  {sid_key} {bkey}: {vals}')

    # Install poppler and display aggregated figures
    sp.run(['apt-get', 'install', '-y', '-q', 'poppler-utils'])
    for pdf in sorted(glob.glob(os.path.join(run_dir, 'figures', '*.pdf'))):
        png_base = pdf.replace('.pdf', '')
        sp.run(['pdftoppm', '-r', '150', '-png', '-singlefile', pdf, png_base])
        png = png_base + '.png'
        if os.path.exists(png):
            img = mpimg.imread(png)
            fig, ax = plt.subplots(figsize=(14, 8))
            ax.imshow(img); ax.axis('off')
            ax.set_title(os.path.basename(pdf), fontsize=10)
            plt.tight_layout(); plt.show()